# Bodhan through the OpenAI SDK

The `/v1` routes on `https://api.bodhan.ai` accept OpenAI-style requests, so the official `openai` package works with just a `base_url`. This notebook covers the three OpenAI-shaped models and shows the two plain-JSON endpoints alongside for completeness.

In [ ]:
%pip install -q openai==1.51.0 requests==2.32.3

In [ ]:
import os, json, requests
from openai import OpenAI

client = OpenAI(base_url="https://api.bodhan.ai/v1", api_key=os.environ["BODHAN_API_KEY"])

## Speech to text — `indic-transcribe`

In [ ]:
with open("../../sample_data/sample_hindi.wav", "rb") as fh:
    t = client.audio.transcriptions.create(model="indic-transcribe", file=fh, language="hi")
t.text

## Text to speech — `indic-speak`

`instructions` is a JSON string carrying `lang` (and optionally `style`).

In [ ]:
speech = client.audio.speech.create(
    model="indic-speak",
    voice="Kavya",
    input="नमस्ते, बोधन कुकबुक में आपका स्वागत है।",
    instructions=json.dumps({"lang": "hi"}),
    response_format="wav",
)
speech.write_to_file("hello.wav")

## Document OCR — `indic-ocr`

In [ ]:
import base64

b64 = base64.b64encode(open("page.png", "rb").read()).decode()  # any PNG/JPEG page
r = client.chat.completions.create(
    model="indic-ocr",
    messages=[{"role": "user", "content": [{"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}"}}]}],
)
print(r.choices[0].message.content)
# Layout blocks are a Bodhan extension outside the OpenAI schema; read them from the raw response:
raw = client.chat.completions.with_raw_response.create(
    model="indic-ocr",
    messages=[{"role": "user", "content": [{"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}"}}]}],
)
raw.http_response.json().get("blocks", [])[:3]

## Translation and transliteration — plain JSON

These endpoints are not OpenAI-shaped. Use `requests` (or any HTTP client) with the same key.

In [ ]:
H = {"Authorization": f"Bearer {os.environ['BODHAN_API_KEY']}", "Content-Type": "application/json"}
print(requests.post("https://api.bodhan.ai/translate", headers=H, json={"text": "Good morning", "target_language": "ta"}).json()["translation"])
print(requests.post("https://api.bodhan.ai/transliterate", headers=H, json={"text": "namaste", "language": "hi", "script": "native"}).json()["transliteration"])

## Node.js

```js
import OpenAI from "openai";
const client = new OpenAI({ baseURL: "https://api.bodhan.ai/v1", apiKey: process.env.BODHAN_API_KEY });
const t = await client.audio.transcriptions.create({ model: "indic-transcribe", file: fs.createReadStream("clip.wav"), language: "hi" });
```